In [1]:
import torch
import torch.nn as nn
import sys
import os
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, Tuple, Dict, Any

# Add current directory to path
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from models.decoder.medgemma_decoder import MedGemmaDecoder
from utils.registry import ModelRegistry
from utils.enums import BridgeName
from utils.files_handler import load_yaml

print("Imports complete.")

/volume/ECG_tokenizer/.venv/lib/python3.12/site-packages/sklearn/utils/_param_validation.py:11: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


Imports complete.


In [ ]:
# --- Load Configuration ---
# Using the champion config as requested for the dataset path
config_path = "config/llm_finetuning/medgemma/medgemma_4b_champion_5M.yaml"
config = load_yaml(config_path)

print(f"Loaded configuration from {config_path}")
train_parquet_path = config.get('train_dataset_path')
print(f"Train Dataset Path: {train_parquet_path}")

if not os.path.exists(train_parquet_path):
    print(f"\u26A0\uFE0F Warning: Parquet file not found at {train_parquet_path}. Checks mounts.")
else:
    print("\u2705 Parquet file found.")
# --- Load Real Sample from Parquet ---

if os.path.exists(train_parquet_path):
    # Read a small sample to avoid loading 5M rows
    df = pd.read_parquet(train_parquet_path)
    
    # Pick a random sample
    sample = df.sample(1).iloc[0]
    
    real_prompt = sample[config.get('prompt_column', 'prompt')]
    real_ecg_path = sample[config.get('signal_path_column', 'waveform_path_psa')]
    real_answer = sample[config.get('answer_column', 'generated_answer')]
    
    print("\n--- Real Sample Data ---")
    print(f"ECG Path: {real_ecg_path}")
    print(f"Prompt (Truncated): {real_prompt[:200]}...")
    print(f"Answer (Truncated): {real_answer[:100]}...")
    
    # Ensure <start_of_image> token is in prompt (MedGemma expects it)
    if "<start_of_image>" not in real_prompt:
         print("\u26A0\uFE0F Warning: <start_of_image> token missing in raw prompt. Adding it manually for test.")
         real_prompt = "<start_of_image> " + real_prompt
else:
    real_prompt = "<start_of_image> Analyze this ECG and look for ST elevation."
    real_ecg_path = "dummy_path.npy"
    print("Using dummy prompt due to missing dataset.")


--- Real Sample Data ---
ECG Path: /media/data1/datasets/MIMIC-IV/adjusted_signals/train/43894693.npy
Prompt (Truncated): Is this a normal ECG?...
Answer (Truncated): No - Abnormal ECG; Pathological findings: ST depression (lateral leads - I, avL, V5-V6)...
⚠️ Warning: <start_of_image> token missing in raw prompt. Adding it manually for test.


In [4]:
# --- Instantiate MedGemmaDecoder with Real Weights ---

# Extract parameters from config
hf_model = config.get('huggingface_model_name', 'google/medgemma-4b-it')
emb_size = config.get('llm_input_embedding_size', 2560)
stage1_ckpt = config.get('stage1_checkpoint_path')
bridge_name = config.get('bridge_name', 'InstructionAwareECGQFormerBridge')

# Bridge specific args
bridge_kwargs = {
    'bridge_mid_dim': config.get('bridge_mid_dim', 768),
    'bridge_num_heads': config.get('bridge_num_heads', 12),
    'bridge_dropout': config.get('bridge_dropout', 0.1),
    'bridge_num_special_tokens': config.get('bridge_num_special_tokens', 4),
    'bridge_qformer_layers': config.get('bridge_qformer_layers', 6),
    'bridge_text_hidden_size': config.get('bridge_text_hidden_size', 768),
    'bridge_bias_last_codebook': config.get('bridge_bias_last_codebook', 0.5),
    'bridge_codebook_dropout': config.get('bridge_codebook_dropout', 0.0),
    'bridge_cross_every': config.get('bridge_cross_every', 2),
    'instruction_dropout': config.get('instruction_dropout', 0.1),
    'num_codebooks_kept': config.get('num_codebooks_kept', 8),
    'ecg_codebook_size': config.get('ecg_codebook_size', 512),
    'num_quantizers': config.get('num_quantizers', 8),
}

print("Initializing MedGemmaDecoder (this may take a while to load weights)...")

try:
    decoder = MedGemmaDecoder(
        huggingface_model_name=hf_model,
        llm_input_embedding_size=emb_size,
        bridge_name=bridge_name,
        stage1_checkpoint_path=stage1_ckpt,
        # For champion config, num_ecg_tokens is 0, which implies dynamic soft tokens
        # But we need to tell the decoder how many visual tokens to expect/use if strictly defined
        # Actually the class uses adapter/bridge num_tokens if num_ecg_tokens is 0/None
        # QFormer output length is fixed (usually 32). Let's trust the bridge default.
        num_ecg_tokens=config.get('num_ecg_tokens', 32) or 32,
        **bridge_kwargs
    )
    print("\n\u2705 Successfully initialized MedGemmaDecoder with real weights!")
    
except Exception as e:
    print(f"\n\u274C Failed to initialize model: {e}")
    import traceback
    traceback.print_exc()

Initializing MedGemmaDecoder (this may take a while to load weights)...


/volume/ECG_tokenizer/models/decoder/medgemma_decoder.py:939: UserWarning: Stage-1 bridge checkpoint 'checkpoints/ECG_Text_Stage1/ecg_text_stage1/BEST_QFORMER_MEDGEMMA_j4bb0w33_20251113-141927/checkpoints/stage1_last_epoch_010.pt' mismatches current config: num_query_tokens (config 32 vs model 128). Proceeding to load whatever keys match.
  warnings.warn(


[Stage1] attempting to load checkpoint for bridge 'InstructionAwareECGQFormerBridge' from 'checkpoints/ECG_Text_Stage1/ecg_text_stage1/BEST_QFORMER_MEDGEMMA_j4bb0w33_20251113-141927/checkpoints/stage1_last_epoch_010.pt'


/volume/ECG_tokenizer/models/decoder/medgemma_decoder.py:994: UserWarning: bridge 'InstructionAwareECGQFormerBridge' Stage-1 checkpoint expects 10 blocks but config instantiated 6. Loading available tensors despite mismatch.
  warnings.warn(
/volume/ECG_tokenizer/models/decoder/medgemma_decoder.py:1002: UserWarning: bridge 'InstructionAwareECGQFormerBridge' Stage-1 checkpoint expects 10 instruction blocks but config instantiated 6. Loading available tensors despite mismatch.
  warnings.warn(


[Stage1] bridge 'InstructionAwareECGQFormerBridge': partially loaded 1 tensors (tokenizer alignment); reinitialised tensors: instruction_norm.weight, norm_out.weight, to_llm.bias, to_llm.weight
[Stage1] checkpoint loaded for bridge 'InstructionAwareECGQFormerBridge': checkpoints/ECG_Text_Stage1/ecg_text_stage1/BEST_QFORMER_MEDGEMMA_j4bb0w33_20251113-141927/checkpoints/stage1_last_epoch_010.pt


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


✅ Successfully initialized MedGemmaDecoder with real weights!


In [5]:
# --- Minimal Sanity Checks: 1 & 2 ---

def run_sanity_checks():
    print("\n=== Starting Sanity Checks ===")
    
    device = decoder.llm_model.device
    
    # --- Check 2: Verify <start_of_image> token presence ---
    # The decoder._start_image_token_id() method retrieves the ID
    image_token_id = decoder._start_image_token_id()
    print(f"\n[Check 2] Image Token ID: {image_token_id}")
    
    # Tokenize the real prompt
    inputs = decoder.tokenizer(real_prompt, return_tensors="pt", add_special_tokens=True)
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)
    
    # Scan for image token
    has_image = (input_ids == image_token_id).any()
    img_indices = (input_ids == image_token_id).nonzero(as_tuple=False)
    
    if has_image:
        print(f"\u2705 <start_of_image> token FOUND at index/indices: {img_indices.tolist()}")
        # Decode around the image token to verify context
        idx = img_indices[0, 1].item()
        start = max(0, idx - 5)
        end = min(input_ids.shape[1], idx + 5)
        context = decoder.tokenizer.decode(input_ids[0, start:end])
        print(f"Context around <start_of_image>: '{context}'")
    else:
        print(f"\u274C <start_of_image> token NOT FOUND in tokenized sequence! Prompt was: '{real_prompt}'")
        print(f"Input IDs: {input_ids[0].tolist()}")
        
    # --- Check 1: Shape Asserts inside Forward ---
    print("\n[Check 1] Verifying Shapes in Forward Pass...")
    
    # Mock codes
    batch_size = input_ids.size(0)
    quantized_codes = torch.randint(
        0, config.get('ecg_codebook_size', 512), 
        (batch_size, 256, 8)
    ).permute(0, 2, 1).to(device)
    
    labels = input_ids.clone()
    
    # Wrapper to capture internals isn't easy without editing code, but we can infer from output or hook
    # Or better: we can manually call the injection method here to verify shapes explicitly
    
    ecg_embeddings, _ = decoder._compute_ecg_embeddings(
        None, quantized_codes, 
        prompt_input_ids=input_ids, 
        prompt_attention_mask=attention_mask
    )
    
    # Run injection
    merged = decoder._inject_ecg_after_image_token(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
        ecg_embeddings=ecg_embeddings,
        embed_layer=decoder.llm_model.get_input_embeddings()
    )
    
    if merged:
        inputs_embeds, attn_mask, new_ids, prepared_labels = merged
        print(f"Final Embeddings Shape: {inputs_embeds.shape}")
        print(f"Final Mask Shape: {attn_mask.shape}")
        print(f"Final Labels Shape: {prepared_labels.shape}")
        
        try:
            assert inputs_embeds.shape[:2] == attn_mask.shape
            if prepared_labels is not None:
                assert prepared_labels.shape == attn_mask.shape
            print("\u2705 Shape Assertions PASSED.")
        except AssertionError as e:
            print("\u274C Shape Assertions FAILED!")
            raise e
    else:
        print("\u274C Injection failed (returned None).")

run_sanity_checks()


=== Starting Sanity Checks ===

[Check 2] Image Token ID: 169
❌ <start_of_image> token NOT FOUND in tokenized sequence! Prompt was: '<start_of_image> Is this a normal ECG?'
Input IDs: [2, 255999, 2375, 672, 496, 3867, 106520, 236881]

[Check 1] Verifying Shapes in Forward Pass...
❌ Injection failed (returned None).


In [6]:
# --- Minimal Sanity Check 3: Tiny Overfit Experiment ---

def run_tiny_overfit():
    print("\n=== [Check 3] Tiny Overfit Experiment ===")
    
    device = decoder.llm_model.device
    decoder.train()
    
    # Freeze LLM to make it faster/easier to overfit just the bridge/adapters if possible
    # Or just train everything. Let's use the config settings: freeze_llm=True usually.
    decoder.freeze_llm_parameters()
    print("LLM frozen, training bridge/adapters only.")
    
    optimizer = torch.optim.AdamW(decoder.parameters(), lr=1e-4)
    
    # Setup ONE fixed sample
    prompt_text = "<start_of_image> Is the rhythm Regular? Answer Yes or No."
    target_text = "Yes"
    
    # Full text: "<start_of_image> ... Yes"
    # We need to construct input_ids containing prompt + answer
    full_text = prompt_text + " " + target_text
    encoding = decoder.tokenizer(full_text, return_tensors="pt", add_special_tokens=True)
    input_ids = encoding.input_ids.to(device)
    attention_mask = encoding.attention_mask.to(device)
    
    # Labels: mask out prompt, keep answer
    # Find where answer starts (simplistic approach)
    labels = input_ids.clone()
    # Mask out everything before the last token(s) corresponding to "Yes"
    # Let's just mask the first N-1 tokens for this simple test
    answer_len = len(decoder.tokenizer.encode(target_text, add_special_tokens=False))
    labels[:, :-answer_len] = -100
    
    print(f"Training Sample: '{full_text}'")
    print(f"Target Answer: '{target_text}'")
    
    # Fixed mock codes
    fixed_codes = torch.randint(
        0, config.get('ecg_codebook_size', 512), 
        (1, 256, 8)
    ).permute(0, 2, 1).to(device)
    
    # Training Loop
    print("Starting loop (20 steps)...")
    for step in range(20):
        optimizer.zero_grad()
        
        outputs = decoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            quantized_codes=fixed_codes,
            prompt_input_ids=input_ids,
            prompt_attention_mask=attention_mask
        )
        
        loss = outputs["loss"]
        loss.backward()
        optimizer.step()
        
        if step % 5 == 0 or step == 19:
            print(f"Step {step}: Loss = {loss.item():.4f}")
            
    print("\nVerifying Generation...")
    decoder.eval()
    with torch.no_grad():
        # Generate using just the prompt
        gen_input = decoder.tokenizer(prompt_text, return_tensors="pt", add_special_tokens=True)
        gen_ids = gen_input.input_ids.to(device)
        gen_mask = gen_input.attention_mask.to(device)
        
        # MedGemma/HF generate
        # We need to pass codes via some mechanism if generate supports it
        # MedGemmaDecoder.generate_report handles this
        generated_ids = decoder.generate_report(
            quantized_codes=fixed_codes,
            input_ids=gen_ids,
            attention_mask=gen_mask,
            max_new_tokens=10
        )
        
        # Decode
        output_text = decoder.tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        print(f"Generated: '{output_text}'")
        
        if target_text.lower() in output_text.lower():
            print("\u2705 Overfit SUCCESS: Generated correct answer.")
        else:
            print("\u274C Overfit FAILED: Did not generate correct answer.")

run_tiny_overfit()


=== [Check 3] Tiny Overfit Experiment ===
LLM frozen, training bridge/adapters only.
Training Sample: '<start_of_image> Is the rhythm Regular? Answer Yes or No. Yes'
Target Answer: 'Yes'
Starting loop (20 steps)...


/volume/ECG_tokenizer/models/decoder/medgemma_decoder.py:1448: UserWarning: LoRA was requested but no parameters matching 'lora_' were found on the MedGemma model.
  warnings.warn(


Step 0: Loss = 6.7111
Step 5: Loss = 5.1482
Step 10: Loss = 2.6156
Step 15: Loss = 1.1409
Step 19: Loss = 0.4898

Verifying Generation...
Generated: 'I am unable to access external websites or specific files'
❌ Overfit FAILED: Did not generate correct answer.
